# TGT — K-Fold Cross-Validation (file-level)

Trains the TGT from scratch on each fold and reports mean ± std metrics across folds.
Splits are at the **file** level so overlapping windows never leak between train and val.

**Runtime → T4 GPU**, then Run All.

In [ ]:
#@title 1. Setup
import os

if not os.path.exists("sebi.py"):
    !git clone https://github.com/sebinkooooo/mech.git
    os.chdir("mech")

%pip install -q torch torchvision torchaudio torchinfo seaborn scikit-learn tqdm matplotlib

HOME = os.getcwd()
print("Working directory:", HOME)

In [ ]:
#@title 2. Hyperparameters
WINDOW_SIZE       = 80
STRIDE            = 30

D_MODEL           = 32
NUM_HEADS         = 2
NUM_LAYERS        = 2
DIM_FEEDFORWARD   = 128
DROPOUT           = 0.3

BATCH_SIZE        = 64
LEARNING_RATE     = 5e-4
WEIGHT_DECAY      = 1e-3
NUM_EPOCHS        = 150
PATIENCE          = 80
LABEL_SMOOTHING   = 0.1
MIXUP_ALPHA       = 0.0      # set to 0.2 to test with mixup
SCHEDULER         = True

STANDARDIZE       = True
NORMALIZE         = False
WINDOW_NORM       = True
AUGMENT           = True

SEED              = 42
K_FOLDS           = 5

TRAIN_DATA_DIR    = os.path.join(HOME, "train_dataset")
GROUP10_DATA_DIR  = os.path.join(HOME, "Group_10")
FIGURES_DIR       = os.path.join(HOME, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
#@title 3. Imports & seeding
import random, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score

from sebi import (
    FEATURE_COLUMNS, LABEL_MAPPING, CLASS_NAMES,
    load_and_window,
    GaitDataset, TGTModel, compute_class_weights, make_sample_weights,
    train_one_epoch, evaluate, predict_all,
)

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
#@title 4. Load & window all data
feat_train, lab_train, fid_train = load_and_window(
    TRAIN_DATA_DIR, FEATURE_COLUMNS, LABEL_MAPPING, WINDOW_SIZE, STRIDE
)
print(f"train_dataset : {feat_train.shape[0]} windows from {len(np.unique(fid_train))} files")

g10_label_map = {k: v for k, v in LABEL_MAPPING.items() if k != "Jumping"}
feat_g10, lab_g10, fid_g10 = load_and_window(
    GROUP10_DATA_DIR, FEATURE_COLUMNS, g10_label_map, WINDOW_SIZE, STRIDE
)
print(f"Group_10      : {feat_g10.shape[0]} windows from {len(np.unique(fid_g10))} files")

fid_g10 = fid_g10 + fid_train.max() + 1
features = np.concatenate([feat_train, feat_g10], axis=0)
labels   = np.concatenate([lab_train, lab_g10], axis=0)
file_ids = np.concatenate([fid_train, fid_g10], axis=0)

print(f"Total: {features.shape[0]} windows, {len(np.unique(file_ids))} files")
print(f"Classes: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=5)))}")

In [ ]:
#@title 5. Build K file-level folds
rng = np.random.RandomState(SEED)
unique_files = np.unique(file_ids)
rng.shuffle(unique_files)

fold_files = np.array_split(unique_files, K_FOLDS)

for i, ff in enumerate(fold_files):
    n_win = np.isin(file_ids, ff).sum()
    print(f"Fold {i}: {len(ff)} files, {n_win} windows")

In [ ]:
#@title 6. Cross-validation loop
from sklearn.metrics import precision_recall_fscore_support

fold_results = []

for fold_i in range(K_FOLDS):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_i + 1} / {K_FOLDS}")
    print(f"{'='*60}")

    seed_everything(SEED + fold_i)

    val_file_set = set(fold_files[fold_i].tolist())
    train_file_set = set()
    for j in range(K_FOLDS):
        if j != fold_i:
            train_file_set.update(fold_files[j].tolist())

    train_mask = np.isin(file_ids, list(train_file_set))
    val_mask   = np.isin(file_ids, list(val_file_set))
    train_idx  = np.where(train_mask)[0]
    val_idx    = np.where(val_mask)[0]

    print(f"  Train: {len(train_idx)} windows ({len(train_file_set)} files)")
    print(f"  Val  : {len(val_idx)} windows ({len(val_file_set)} files)")

    # datasets
    train_ds = GaitDataset(features[train_idx], labels[train_idx],
                           standardize=STANDARDIZE, normalize=NORMALIZE,
                           window_norm=WINDOW_NORM, augment=AUGMENT)
    val_ds = GaitDataset(features[val_idx], labels[val_idx],
                         scaler=train_ds.scaler, window_norm=WINDOW_NORM)

    sample_weights = make_sample_weights(labels[train_idx])
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    # fresh model
    num_features = features.shape[2]
    model = TGTModel(
        num_features=num_features, window_size=WINDOW_SIZE, num_classes=5,
        d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT,
    ).to(device)

    class_weights = compute_class_weights(labels[train_idx], num_classes=5).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS) if SCHEDULER else None

    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device,
                                        mixup_alpha=MIXUP_ALPHA)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        if scheduler:
            scheduler.step()

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 25 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{NUM_EPOCHS} │ "
                  f"Train {t_loss:.4f}/{t_acc:.1f}% │ "
                  f"Val {v_loss:.4f}/{v_acc:.1f}%")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stop at epoch {epoch}")
            break

    # load best checkpoint and evaluate
    model.load_state_dict(best_state)
    model.to(device)
    v_loss, v_acc = evaluate(model, val_loader, criterion, device)
    preds, true = predict_all(model, val_loader, device)

    present = sorted(set(true.tolist()))
    prec, rec, f1, sup = precision_recall_fscore_support(true, preds, labels=present, zero_division=0)
    macro_f1 = f1_score(true, preds, labels=present, average="macro", zero_division=0)

    print(f"\n  Fold {fold_i+1} best val acc: {v_acc:.2f}%  |  macro F1: {macro_f1:.4f}")
    for ci, c in enumerate(present):
        print(f"    {CLASS_NAMES[c]:20s}  P={prec[ci]:.2f}  R={rec[ci]:.2f}  F1={f1[ci]:.2f}  n={sup[ci]}")

    fold_results.append({
        "fold": fold_i + 1,
        "val_acc": v_acc,
        "val_loss": v_loss,
        "macro_f1": macro_f1,
        "per_class_f1": {CLASS_NAMES[c]: f1[ci] for ci, c in enumerate(present)},
        "per_class_prec": {CLASS_NAMES[c]: prec[ci] for ci, c in enumerate(present)},
        "per_class_rec": {CLASS_NAMES[c]: rec[ci] for ci, c in enumerate(present)},
    })

print(f"\n{'='*60}")
print("  ALL FOLDS COMPLETE")
print(f"{'='*60}")

In [ ]:
#@title 7. Aggregate results
accs     = [r["val_acc"] for r in fold_results]
losses   = [r["val_loss"] for r in fold_results]
macro_f1s = [r["macro_f1"] for r in fold_results]

print("Per-fold results:")
print(f"{'Fold':>6}  {'Acc':>8}  {'Loss':>8}  {'Macro F1':>10}")
print("-" * 40)
for r in fold_results:
    print(f"{r['fold']:>6}  {r['val_acc']:>7.2f}%  {r['val_loss']:>8.4f}  {r['macro_f1']:>10.4f}")

print("-" * 40)
print(f"{'Mean':>6}  {np.mean(accs):>7.2f}%  {np.mean(losses):>8.4f}  {np.mean(macro_f1s):>10.4f}")
print(f"{'Std':>6}  {np.std(accs):>7.2f}%  {np.std(losses):>8.4f}  {np.std(macro_f1s):>10.4f}")

print(f"\nOverall: {np.mean(accs):.2f} ± {np.std(accs):.2f}% accuracy")
print(f"         {np.mean(macro_f1s):.4f} ± {np.std(macro_f1s):.4f} macro F1")

In [ ]:
#@title 8. Per-class F1 across folds
print(f"\n{'Class':>20}  {'Mean F1':>8}  {'Std F1':>8}  {'Min':>6}  {'Max':>6}")
print("-" * 58)
for cls_name in CLASS_NAMES:
    class_f1s = [r["per_class_f1"].get(cls_name, float('nan')) for r in fold_results]
    valid = [x for x in class_f1s if not np.isnan(x)]
    if valid:
        print(f"{cls_name:>20}  {np.mean(valid):>8.4f}  {np.std(valid):>8.4f}  "
              f"{np.min(valid):>6.2f}  {np.max(valid):>6.2f}")
    else:
        print(f"{cls_name:>20}  {'n/a':>8}")

In [ ]:
#@title 9. Bar chart — per-class mean F1 ± std
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

class_means, class_stds, class_labels = [], [], []
for cls_name in CLASS_NAMES:
    vals = [r["per_class_f1"].get(cls_name, float('nan')) for r in fold_results]
    valid = [x for x in vals if not np.isnan(x)]
    if valid:
        class_means.append(np.mean(valid))
        class_stds.append(np.std(valid))
        class_labels.append(cls_name)

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(class_labels))
bars = ax.bar(x, class_means, yerr=class_stds, capsize=5,
              color="steelblue", edgecolor="black", alpha=0.85)
for b, m, s in zip(bars, class_means, class_stds):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + s + 0.01,
            f"{m:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=20, ha="right")
ax.set_ylim(0, 1.15)
ax.set_ylabel("F1 Score")
ax.set_title(f"{K_FOLDS}-Fold Cross-Val: Per-Class F1 (mean ± std)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_path = os.path.join(FIGURES_DIR, "crossval_per_class_f1.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")